<a href="https://colab.research.google.com/github/alilouati/data-visualization-using-python/blob/main/Chapter_3/Ch3_3D_and_Multivariate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 3: 3-D Plotting & Multivariate Visualization
**Data Visualization — Ph.D. in Computing** · Dr. Ali Louati · IS Dept., PSAU

⏱️ Estimated lab time: **2 × 90 min**

### Learning objectives
By the end of this lab you should be able to:
1. Create 3-D scatter, surface and wireframe plots with `mpl_toolkits.mplot3d` and interactive ones with Plotly.
2. Visualise optimisation landscapes (saddle points, local optima) and overlay optimiser trajectories.
3. Argue when 3-D helps and when contours, heatmaps or small multiples are more accurate (occlusion, perspective).
4. Render real elevation data (Maunga Whau volcano) and 3-D scalar fields (isosurfaces).
5. Visualise high-dimensional data via 2-D/3-D embeddings (PCA, t-SNE) and clustered heatmaps.

### How to use this notebook
* Run cells top to bottom with **Shift + Enter** (or *Runtime → Run all*).
* Cells marked **✏️ Exercise** are for you. Cells marked **🔑 Solution** are collapsed: try first, then double-click to reveal.
* Save your own copy: *File → Save a copy in Drive*.

## 0. Setup

In [ ]:
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px, plotly.graph_objects as go
from plotly.subplots import make_subplots
IN_COLAB = "google.colab" in sys.modules
sns.set_theme(style="white")
rng = np.random.default_rng(3)

## 1. 3-D scatter with Matplotlib (`mpl_toolkits.mplot3d`)
Since Matplotlib 3.2 you only need `projection="3d"`; importing `Axes3D` is no longer required.

In [ ]:
tips = sns.load_dataset("tips")
fig = plt.figure(figsize=(11, 4.2))
for i, (elev, azim) in enumerate([(20, -60), (20, 30)], start=1):
    ax = fig.add_subplot(1, 2, i, projection="3d")
    sc = ax.scatter(tips.total_bill, tips["size"], tips.tip, c=tips.tip, cmap="viridis", s=18, depthshade=True)
    ax.set(xlabel="total bill", ylabel="party size", zlabel="tip", title=f"view elev={elev}, azim={azim}")
    ax.view_init(elev, azim)
fig.colorbar(sc, ax=fig.axes, shrink=.6, label="tip"); plt.show()

> **Observe.** The *same* data looks different from two viewpoints, and depth can't be read reliably from a static image. Static 3-D scatters need either interaction or projections.

In [ ]:
fig = px.scatter_3d(tips, x="total_bill", y="size", z="tip", color="time", symbol="smoker", opacity=.8, height=480,
                    title="Interactive 3-D scatter — drag to rotate")
fig.update_traces(marker_size=4); fig.show()

## 2. Functions of two variables: surfaces, wireframes, contours
### 2.1 Saddle point  $f(x,y)=x^2-y^2$
At (0,0) the gradient is zero, but it is a minimum along *x* and a maximum along *y*. The Hessian has eigenvalues of opposite sign.

In [ ]:
x = y = np.linspace(-2, 2, 80); X, Y = np.meshgrid(x, y)
Z = X**2 - Y**2
fig = plt.figure(figsize=(15, 4))
ax1 = fig.add_subplot(1, 3, 1, projection="3d"); ax1.plot_surface(X, Y, Z, cmap="coolwarm", linewidth=0, antialiased=True); ax1.set_title("surface")
ax2 = fig.add_subplot(1, 3, 2, projection="3d"); ax2.plot_wireframe(X, Y, Z, rstride=6, cstride=6, color="k", lw=.5); ax2.set_title("wireframe")
ax3 = fig.add_subplot(1, 3, 3); cs = ax3.contourf(X, Y, Z, levels=20, cmap="coolwarm"); ax3.contour(X, Y, Z, levels=[0], colors="k")
ax3.plot(0, 0, "k*", ms=12); ax3.set_aspect("equal"); ax3.set_title("filled contour (zero level in black)"); fig.colorbar(cs, ax=ax3)
plt.tight_layout(); plt.show()
print("Hessian at origin:", np.array([[2, 0], [0, -2]]).tolist(), "→ eigenvalues", np.linalg.eigvalsh([[2, 0], [0, -2]]))

### 2.2 Local vs global optima: Himmelblau and Rastrigin
$f_\text{H}(x,y)=(x^2+y-11)^2+(x+y^2-7)^2$ has **four** global minima.
Rastrigin $f_\text{R}(x,y)=20+\sum (x_i^2-10\cos 2\pi x_i)$ has a single global minimum surrounded by a lattice of local minima — a classic stress test for optimisers.

In [ ]:
himmelblau = lambda x, y: (x**2 + y - 11)**2 + (x + y**2 - 7)**2
rastrigin  = lambda x, y: 20 + x**2 - 10*np.cos(2*np.pi*x) + y**2 - 10*np.cos(2*np.pi*y)

fig = make_subplots(rows=1, cols=2, specs=[[{"type": "surface"}, {"type": "surface"}]],
                    subplot_titles=["Himmelblau (4 global minima)", "Rastrigin (many local minima)"])
g = np.linspace(-5, 5, 120); GX, GY = np.meshgrid(g, g)
fig.add_trace(go.Surface(x=g, y=g, z=np.log1p(himmelblau(GX, GY)), colorscale="Viridis", showscale=False), 1, 1)
r = np.linspace(-5.12, 5.12, 150); RX, RY = np.meshgrid(r, r)
fig.add_trace(go.Surface(x=r, y=r, z=rastrigin(RX, RY), colorscale="Viridis", showscale=False), 1, 2)
fig.update_layout(height=450, margin=dict(l=0, r=0, t=40, b=0)); fig.show()

### 2.3 Research use-case: visualising optimiser trajectories
Contours + trajectories are the standard way to compare optimisers in ML papers. Here: gradient descent vs momentum vs Adam on Himmelblau from the same start.

In [ ]:
def grad_h(p):
    x, y = p
    return np.array([4*x*(x**2 + y - 11) + 2*(x + y**2 - 7), 2*(x**2 + y - 11) + 4*y*(x + y**2 - 7)])

def run(opt, p0=(-0.5, -4.5), lr=0.01, steps=300):
    p, v, m, s, path = np.array(p0, float), np.zeros(2), np.zeros(2), np.zeros(2), [p0]
    for t in range(1, steps + 1):
        g = grad_h(p)
        if opt == "GD":         p = p - lr * g
        elif opt == "Momentum": v = 0.9 * v + g; p = p - lr * v
        elif opt == "Adam":
            m = 0.9*m + 0.1*g; s = 0.999*s + 0.001*g**2
            p = p - 0.1 * (m/(1-0.9**t)) / (np.sqrt(s/(1-0.999**t)) + 1e-8)
        path.append(tuple(p))
    return np.array(path)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
cs = ax.contour(GX, GY, np.log1p(himmelblau(GX, GY)), levels=25, cmap="Greys")
for opt, c in [("GD", "C0"), ("Momentum", "C1"), ("Adam", "C3")]:
    P = run(opt, lr=0.002 if opt == "Momentum" else 0.01)
    ax.plot(P[:, 0], P[:, 1], "-o", ms=2, color=c, label=f"{opt} → ({P[-1,0]:.2f}, {P[-1,1]:.2f})")
ax.plot(-0.5, -4.5, "ks", label="start"); ax.set_aspect("equal"); ax.legend(fontsize=8, loc="upper left")
ax.set_title("Optimiser paths on log(1 + Himmelblau)"); plt.show()

### ✏️ Exercise 1
1. Run the three optimisers on **Rastrigin** from 20 random starts. Plot the end points on the contour map, coloured by optimiser.
2. Make a 3-D Plotly surface with the trajectory drawn as a `go.Scatter3d` line *on* the surface (z = f(x,y) + small offset).
3. Which view (contour or 3-D) lets a reader compare the three optimisers more accurately? Justify with perception arguments from Chapter 1.

In [ ]:
# ✏️ Your code here

In [ ]:
#@title 🔑 Solution (part 2) (try first, then double-click to reveal) { display-mode: "form" }
P = run("Adam")
fig = go.Figure([go.Surface(x=g, y=g, z=np.log1p(himmelblau(GX, GY)), colorscale="Viridis", opacity=.85, showscale=False),
                 go.Scatter3d(x=P[:, 0], y=P[:, 1], z=np.log1p(himmelblau(P[:, 0], P[:, 1])) + 0.05,
                              mode="lines+markers", marker=dict(size=2, color="red"), line=dict(color="red", width=5))])
fig.update_layout(height=480, title="Adam on Himmelblau"); fig.show()

## 3. Real elevation data: the Maunga Whau volcano (Auckland)
⚠️ *Correction to the original slides:* the Plotly "Mt Bruno elevation" sample is a different dataset. The real Maunga Whau (Mt Eden) data is R's `volcano` matrix (87 × 61 grid, 10 m spacing), which Plotly also hosts.

In [ ]:
volcano = pd.read_csv("https://raw.githubusercontent.com/plotly/datasets/master/volcano.csv").values
print("grid:", volcano.shape, "| elevation range (m):", volcano.min(), "→", volcano.max())
yy, xx = np.arange(volcano.shape[0]) * 10, np.arange(volcano.shape[1]) * 10
fig = go.Figure(go.Surface(z=volcano, x=xx, y=yy, colorscale="Earth",
                           contours={"z": {"show": True, "usecolormap": True, "project_z": True}}))
fig.update_layout(title="Maunga Whau (Mt Eden) — drag to rotate", height=520,
                  scene=dict(xaxis_title="east (m)", yaxis_title="north (m)", zaxis_title="elevation (m)",
                             aspectratio=dict(x=1, y=87/61, z=0.5)))
fig.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
im = axes[0].imshow(volcano, cmap="terrain", origin="lower", extent=[0, xx[-1], 0, yy[-1]])
fig.colorbar(im, ax=axes[0], label="m"); axes[0].set_title("Heatmap (exact values, no occlusion)")
c = axes[1].contour(xx, yy, volcano, levels=np.arange(90, 200, 10), cmap="terrain")
axes[1].clabel(c, fmt="%d", fontsize=7); axes[1].set_aspect("equal"); axes[1].set_title("Topographic contours (10 m)")
plt.tight_layout(); plt.show()

> **3-D or not?** 3-D surfaces convey *shape* and are engaging, but suffer from **occlusion** (hidden regions), **perspective distortion** (equal heights look unequal at different depths) and **viewpoint dependence**. For reading values, contours and heatmaps are more accurate. Use 3-D when the reader can rotate it or when the gestalt shape is the message.

## 4. Truly volumetric data: isosurfaces
A scalar field $f(x,y,z)$ (e.g. medical CT, electron density, simulation output) can't be shown as a surface. An **isosurface** draws where $f = c$.

In [ ]:
n = 30
X3, Y3, Z3 = np.mgrid[-2:2:n*1j, -2:2:n*1j, -2:2:n*1j]
F3 = np.sin(np.pi*X3) * np.cos(np.pi*Y3) * np.sin(np.pi*Z3) + 0.3 * (X3**2 + Y3**2 + Z3**2)
fig = go.Figure(go.Isosurface(x=X3.ravel(), y=Y3.ravel(), z=Z3.ravel(), value=F3.ravel(),
                              isomin=0.4, isomax=1.6, surface_count=3, colorscale="Plasma",
                              caps=dict(x_show=False, y_show=False, z_show=False), opacity=.6))
fig.update_layout(height=480, title="Isosurfaces of a 3-D scalar field"); fig.show()

## 5. Beyond three variables
### 5.1 Violin plots (density per group)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
sns.violinplot(data=tips, x="day", y="total_bill", hue="sex", split=True, inner="quart", ax=axes[0])
axes[0].set_title("Tips: split violins by sex")
iris = sns.load_dataset("iris")
sns.violinplot(data=iris.melt(id_vars="species"), x="variable", y="value", hue="species", ax=axes[1], inner=None, cut=0)
axes[1].set_title("Iris: all four measurements"); axes[1].tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()

### 5.2 Heatmaps: flights (sequential) and correlation (diverging, centred at 0)

In [ ]:
flights = sns.load_dataset("flights").pivot(index="month", columns="year", values="passengers")
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
sns.heatmap(flights, cmap="rocket_r", annot=True, fmt="d", annot_kws={"size": 6}, cbar_kws={"label": "passengers"}, ax=axes[0])
axes[0].set_title("Flights: month × year")
corr = sns.load_dataset("mpg").select_dtypes("number").corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap="vlag", center=0, vmin=-1, vmax=1, annot=True, fmt=".2f", square=True, ax=axes[1])
axes[1].set_title("Correlation (diverging map, centre = 0)")
plt.tight_layout(); plt.show()

In [ ]:
# Clustered heatmap: rows/columns re-ordered by hierarchical clustering reveal structure
sns.clustermap(flights, cmap="rocket_r", standard_scale=1, figsize=(7, 6), method="average")
plt.show()

### 5.3 Dimensionality reduction: seeing 64-D data
The `digits` dataset has 64 pixel features. Compare a **linear** (PCA) and **non-linear** (t-SNE) projection in 3-D.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
digits = load_digits(); Xd, yd = digits.data, digits.target.astype(str)
pca3 = PCA(n_components=3, random_state=0).fit(Xd)
P3 = pca3.transform(Xd)
T3 = TSNE(n_components=3, perplexity=30, init="pca", random_state=0).fit_transform(Xd)
print("PCA explained variance (3 comps):", pca3.explained_variance_ratio_.round(3), "→ total", pca3.explained_variance_ratio_.sum().round(3))
for name, E in [("PCA", P3), ("t-SNE", T3)]:
    fig = px.scatter_3d(x=E[:, 0], y=E[:, 1], z=E[:, 2], color=yd, opacity=.7, height=430,
                        title=f"Digits in 3-D — {name}", category_orders={"color": [str(i) for i in range(10)]})
    fig.update_traces(marker_size=2.5); fig.show()

> **Caution (PhD level).** t-SNE distances *between* clusters and cluster *sizes* are not meaningful; results depend on perplexity and seed (Wattenberg et al., 2016). Always report hyper-parameters and try several seeds.

### ✏️ Exercise 2
Run t-SNE in 2-D with perplexity ∈ {5, 30, 100} and two seeds each. Show the 6 results as small multiples. Write three sentences on what is stable and what is not. *(Bonus: compute trustworthiness with `sklearn.manifold.trustworthiness`.)*

In [ ]:
# ✏️ Your code here

In [ ]:
#@title 🔑 Solution (try first, then double-click to reveal) { display-mode: "form" }
from sklearn.manifold import trustworthiness
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
for j, perp in enumerate([5, 30, 100]):
    for i, seed in enumerate([0, 1]):
        E = TSNE(n_components=2, perplexity=perp, random_state=seed, init="pca").fit_transform(Xd)
        axes[i, j].scatter(E[:, 0], E[:, 1], c=digits.target, cmap="tab10", s=3)
        axes[i, j].set_title(f"perp={perp}, seed={seed}, T={trustworthiness(Xd, E, n_neighbors=10):.3f}", fontsize=9)
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
plt.tight_layout(); plt.show()

## 6. ✏️ Take-home assignment
Choose either (a) a loss landscape from your own research model (e.g. along two random directions, Li et al., 2018) or (b) a 3-D scientific dataset.
Produce **one** interactive 3-D view and **one** 2-D alternative showing the same information. In 300 words, evaluate both against: accuracy of value reading, discoverability of structure, and suitability for print.

## Quick check
1. What do the Hessian eigenvalues tell you at a saddle point?
2. Give two perceptual problems of static 3-D surfaces.
3. When should a heatmap use a diverging colour map?
4. Why can't you interpret inter-cluster distances in t-SNE?

## References
* Li, H. et al. (2018). *Visualizing the Loss Landscape of Neural Nets*. NeurIPS.
* van der Maaten, L. & Hinton, G. (2008). *Visualizing Data using t-SNE*. JMLR.
* Wattenberg, M., Viégas, F., Johnson, I. (2016). *How to Use t-SNE Effectively*. Distill.
* Munzner, T. (2014). *Visualization Analysis and Design*, ch. 6 "Rules of Thumb" (no unjustified 3-D).